# Modelo predictivo de calidad del aire · Estación CEN

Este notebook reproduce el modelo predictivo del dashboard **Aimaj CEN**.

**Enfoque**
- Variable objetivo configurable: PM₁₀, PM₂.₅, O₃, CO8 o temperatura.
- Modelo: regresión Ridge con un modelo directo para cada horizonte: +1 h, +2 h y +3 h.
- Entrenamiento: todos los años disponibles excepto el año base.
- Prueba: el año base completo.
- Métricas: MAE, RMSE y R².

**Datos**
Se leen los archivos `Datos/datos_AAAA.js`, generados por el notebook de limpieza.  
Si ejecutas este notebook desde `Notebooks py`, la ruta esperada es `../Datos`.


## 0. Requisitos

Requiere `pandas`, `numpy`, `scikit-learn` y `matplotlib`.  
En Google Colab puedes instalarlos con:

```python
!pip install -q pandas numpy scikit-learn matplotlib
```


In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
plt.rcParams["figure.figsize"] = (13, 4)


## 1. Configuración

In [ ]:
TARGET = "PM10"          # PM10, PM25, O3, CO o TEMP
BASE_YEAR = 2024         # año que se usa como prueba
HORIZONS = [1, 2, 3]     # horas hacia adelante
ALPHA = 1.0              # regularización Ridge

TARGETS = {
    "PM10": {"key": "pm10", "label": "PM₁₀", "unit": "µg/m³"},
    "PM25": {"key": "pm25", "label": "PM₂.₅", "unit": "µg/m³"},
    "O3": {"key": "o3", "label": "O₃", "unit": "ppm"},
    "CO": {"key": "co8", "label": "CO8", "unit": "ppm"},
    "TEMP": {"key": "temp", "label": "Temperatura", "unit": "°C"},
}

FEATURE_LABELS = [
    "valor actual",
    "rezago 1 h",
    "rezago 2 h",
    "rezago 3 h",
    "media móvil 6 h",
    "media móvil 24 h",
    "temperatura",
    "humedad relativa",
    "velocidad del viento",
    "hora (seno)",
    "hora (coseno)",
    "mes (seno)",
    "mes (coseno)",
]
FEATURE_COLS = [
    "actual",
    "lag1",
    "lag2",
    "lag3",
    "ma6",
    "ma24",
    "temp",
    "rh",
    "wsp",
    "hora_sin",
    "hora_cos",
    "mes_sin",
    "mes_cos",
]

assert len(FEATURE_LABELS) == len(FEATURE_COLS)
print(f"Variable objetivo: {TARGETS[TARGET]['label']} ({TARGETS[TARGET]['unit']})")
print(f"Año base de prueba: {BASE_YEAR}")


## 2. Carga y normalización

Se replican las reglas del dashboard:

- 2015–2022: PM₁₀ y PM₂.₅ en las columnas crudas; RH en la columna 8 y dirección del viento en la 9.
- 2023–2024: PM₁₀ y PM₂.₅ NowCast; RH en la columna 9 y dirección del viento en la 8.
- CO8: media móvil de 8 horas con al menos 6 lecturas; en 2023–2024 se usa el CO8 precalculado cuando hace falta.


In [ ]:
def localizar_datos():
    cwd = Path.cwd()
    candidatos = [cwd / "Datos", cwd.parent / "Datos"]
    for carpeta in candidatos:
        if carpeta.is_dir():
            return carpeta
    raise FileNotFoundError(
        "No encontré la carpeta Datos. Ejecuta el notebook desde "
        "'Notebooks py' o desde la raíz del proyecto, o asigna DATA_DIR manualmente."
    )


DATA_DIR = localizar_datos()
print("Carpeta de datos:", DATA_DIR.resolve())


def cargar_js(ruta):
    texto = ruta.read_text(encoding="utf-8")
    match = re.search(r"=\s*(\{.*\})\s*;?\s*$", texto, re.S)
    if not match:
        raise ValueError(f"No se pudo leer el objeto JavaScript de {ruta.name}")
    return json.loads(match.group(1))


def a_numero(serie):
    return pd.to_numeric(serie, errors="coerce")


def construir_horario(raw):
    year = int(raw["year"])
    arr = pd.DataFrame(raw["h"], columns=[f"c{i}" for i in range(13)])

    if year >= 2023:
        df = pd.DataFrame(
            {
                "o3": a_numero(arr["c0"]),
                "co": a_numero(arr["c1"]),
                "pm10": a_numero(arr["c4"]),
                "pm25": a_numero(arr["c5"]),
                "co8_raw": a_numero(arr["c6"]),
                "temp": a_numero(arr["c7"]),
                "wds": a_numero(arr["c8"]),
                "rh": a_numero(arr["c9"]),
                "wsp": a_numero(arr["c10"]),
            }
        )
        df["co8"] = df["co"].rolling(8, min_periods=6).mean().fillna(df["co8_raw"])
        df = df.drop(columns="co8_raw")
    else:
        df = pd.DataFrame(
            {
                "o3": a_numero(arr["c0"]),
                "co": a_numero(arr["c1"]),
                "pm10": a_numero(arr["c4"]),
                "pm25": a_numero(arr["c5"]),
                "temp": a_numero(arr["c7"]),
                "rh": a_numero(arr["c8"]),
                "wds": a_numero(arr["c9"]),
                "wsp": a_numero(arr["c10"]),
            }
        )
        df["co8"] = df["co"].rolling(8, min_periods=6).mean()

    df["date"] = pd.date_range(f"{year}-01-01", periods=len(df), freq="h")
    df["month"] = df["date"].dt.month
    df["hour"] = df["date"].dt.hour
    return df


DATA = {}
for ruta in sorted(DATA_DIR.glob("datos_*.js")):
    raw = cargar_js(ruta)
    DATA[int(raw["year"])] = construir_horario(raw)

ANIOS = sorted(DATA)
print(f"Años cargados: {ANIOS[0]}–{ANIOS[-1]} ({len(ANIOS)} años)")


In [ ]:
filas = []
for year in ANIOS:
    df = DATA[year]
    filas.append(
        {
            "Año": year,
            "Horas": len(df),
            "PM10 %": df["pm10"].notna().mean() * 100,
            "PM25 %": df["pm25"].notna().mean() * 100,
            "O3 %": df["o3"].notna().mean() * 100,
            "CO8 %": df["co8"].notna().mean() * 100,
            "Temp %": df["temp"].notna().mean() * 100,
            "HR %": df["rh"].notna().mean() * 100,
            "Viento %": df["wsp"].notna().mean() * 100,
        }
    )

resumen_datos = pd.DataFrame(filas).set_index("Año").round(1)
display(resumen_datos)


## 3. Variables de entrada

Cada origen usa únicamente información disponible hasta la hora actual:

- valor actual y rezagos de 1, 2 y 3 horas;
- medias móviles de 6 y 24 horas;
- temperatura, humedad relativa y velocidad del viento;
- codificación cíclica de la hora y del mes.

Las últimas 4 horas de cada año se excluyen para evitar bordes sin futuro conocido.


In [ ]:
ORIGENES_CACHE = {}


def construir_origenes(df, key):
    s = df[key]
    f = pd.DataFrame(index=df.index)
    f["actual"] = s
    f["lag1"] = s.shift(1)
    f["lag2"] = s.shift(2)
    f["lag3"] = s.shift(3)
    f["ma6"] = s.rolling(6, min_periods=3).mean()
    f["ma24"] = s.rolling(24, min_periods=12).mean()
    f["temp"] = df["temp"]
    f["rh"] = df["rh"]
    f["wsp"] = df["wsp"]
    f["hora_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    f["hora_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    f["mes_sin"] = np.sin(2 * np.pi * (df["month"] - 1) / 12)
    f["mes_cos"] = np.cos(2 * np.pi * (df["month"] - 1) / 12)

    f = f.iloc[3:-4].copy()
    f = f.dropna(subset=["actual", "lag1", "lag2", "lag3", "ma6", "ma24"])

    out = f.copy()
    out["date"] = df.loc[out.index, "date"]
    out["hour"] = df.loc[out.index, "hour"]
    out["month"] = df.loc[out.index, "month"]
    for h in HORIZONS:
        out[f"y{h}"] = df[key].shift(-h).loc[out.index]
    return out.reset_index(drop=True)


def origenes(year, key):
    cache_key = (year, key)
    if cache_key not in ORIGENES_CACHE:
        ORIGENES_CACHE[cache_key] = construir_origenes(DATA[year], key)
    return ORIGENES_CACHE[cache_key]


conteo = pd.DataFrame(
    {
        f"{TARGETS[t]['label']}": {year: len(origenes(year, TARGETS[t]["key"])) for year in ANIOS}
        for t in TARGETS
    }
)
display(conteo)


## 4. Entrenamiento y evaluación

In [ ]:
def ajustar_escalador(X):
    medias = np.zeros(X.shape[1])
    desv = np.ones(X.shape[1])
    for j in range(X.shape[1]):
        col = X[:, j]
        validos = col[~np.isnan(col)]
        if validos.size:
            medias[j] = validos.mean()
            sd = validos.std(ddof=0)
            desv[j] = sd if sd > 0 else 1.0
    return medias, desv


def escalar(X, medias, desv):
    Z = (X - medias) / desv
    return np.nan_to_num(Z, nan=0.0)


def calcular_metricas(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    error = y_true - y_pred
    mae = float(np.mean(np.abs(error)))
    rmse = float(np.sqrt(np.mean(error**2)))
    ss_res = float(np.sum(error**2))
    ss_tot = float(np.sum((y_true - y_true.mean()) ** 2))
    r2 = float(1 - ss_res / ss_tot) if ss_tot > 0 else float("nan")
    return mae, rmse, r2


def entrenar_modelo(target, base_year, alpha=ALPHA):
    if target not in TARGETS:
        raise ValueError(f"Variable objetivo no reconocida: {target}")
    if base_year not in DATA:
        raise ValueError(f"El año base {base_year} no está cargado")

    key = TARGETS[target]["key"]
    train_years = [year for year in ANIOS if year != base_year]
    train = pd.concat([origenes(year, key) for year in train_years], ignore_index=True)
    test = origenes(base_year, key).copy()

    resultado = {
        "target": target,
        "base_year": base_year,
        "train_years": train_years,
        "n_train_origenes": len(train),
        "n_test_origenes": len(test),
        "horizontes": {},
    }

    if train.empty or test.empty:
        resultado["error"] = "No hay suficientes orígenes para entrenar/evaluar."
        return resultado

    X_train_all = train[FEATURE_COLS].to_numpy(dtype=float)
    medias, desv = ajustar_escalador(X_train_all)

    for h in HORIZONS:
        tr = train[train[f"y{h}"].notna()].copy()
        te = test[test[f"y{h}"].notna()].copy()

        if len(tr) < 100 or len(te) < 10:
            resultado["horizontes"][h] = {
                "error": "Muestras insuficientes para este horizonte."
            }
            continue

        X_tr = escalar(tr[FEATURE_COLS].to_numpy(dtype=float), medias, desv)
        y_tr = tr[f"y{h}"].to_numpy(dtype=float)
        X_te = escalar(te[FEATURE_COLS].to_numpy(dtype=float), medias, desv)
        y_te = te[f"y{h}"].to_numpy(dtype=float)

        modelo = Ridge(alpha=alpha, fit_intercept=True)
        modelo.fit(X_tr, y_tr)

        # Igual que el dashboard: los contaminantes no pueden ser negativos.
        pred = np.maximum(0.0, modelo.predict(X_te))

        mae, rmse, r2 = calcular_metricas(y_te, pred)

        importancia = pd.DataFrame(
            {"Variable": FEATURE_LABELS, "Coeficiente": modelo.coef_}
        )
        importancia["abs"] = importancia["Coeficiente"].abs()
        importancia = (
            importancia.sort_values("abs", ascending=False)
            .drop(columns="abs")
            .head(5)
            .reset_index(drop=True)
        )

        detalle = pd.DataFrame(
            {
                "date": te["date"].to_numpy(),
                "hour": te["hour"].to_numpy(),
                "obs": y_te,
                "pred": pred,
                "error": np.abs(y_te - pred),
            }
        )

        resultado["horizontes"][h] = {
            "mae": mae,
            "rmse": rmse,
            "r2": r2,
            "n_train": len(tr),
            "n_test": len(te),
            "importancia": importancia,
            "detalle": detalle,
        }

    return resultado


## 5. Resultado para la variable seleccionada

In [ ]:
modelo = entrenar_modelo(TARGET, BASE_YEAR)

if "error" in modelo:
    display(Markdown(f"**No fue posible entrenar:** {modelo['error']}"))
else:
    display(
        Markdown(
            f"**{TARGETS[TARGET]['label']}** · año base **{BASE_YEAR}** · "
            f"entrenamiento {modelo['train_years'][0]}–{modelo['train_years'][-1]}"
        )
    )
    filas = []
    for h, det in modelo["horizontes"].items():
        if "error" in det:
            filas.append(
                {
                    "Horizonte": f"+{h} h",
                    "MAE": np.nan,
                    "RMSE": np.nan,
                    "R²": np.nan,
                    "Entrenamiento": np.nan,
                    "Prueba": np.nan,
                    "Nota": det["error"],
                }
            )
        else:
            filas.append(
                {
                    "Horizonte": f"+{h} h",
                    "MAE": det["mae"],
                    "RMSE": det["rmse"],
                    "R²": det["r2"],
                    "Entrenamiento": det["n_train"],
                    "Prueba": det["n_test"],
                    "Nota": "",
                }
            )
    display(pd.DataFrame(filas).set_index("Horizonte").round(3))


In [ ]:
if "error" not in modelo:
    fig, axes = plt.subplots(
        len(HORIZONS), 1, figsize=(14, 3.4 * len(HORIZONS)), sharex=False
    )
    if len(HORIZONS) == 1:
        axes = [axes]

    for ax, h in zip(axes, HORIZONS):
        det = modelo["horizontes"].get(h)
        if not det or "error" in det:
            ax.axis("off")
            ax.set_title(f"+{h} h · sin datos suficientes")
            continue

        d = det["detalle"]
        paso = max(1, len(d) // 240)
        d = d.iloc[::paso]

        ax.plot(d["date"], d["obs"], color="#4CC9F0", lw=1, label="Observado")
        ax.plot(d["date"], d["pred"], color="#FF7A2F", lw=1, label=f"Predicho +{h} h")
        ax.set_title(
            f"{TARGETS[TARGET]['label']} · +{h} h · "
            f"MAE {det['mae']:.2f} · RMSE {det['rmse']:.2f} · R² {det['r2']:.3f}"
        )
        ax.set_ylabel(TARGETS[TARGET]["unit"])
        ax.grid(alpha=0.2)
        ax.legend(loc="upper right")

    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()


In [ ]:
if "error" not in modelo:
    tablas = []
    for h in HORIZONS:
        det = modelo["horizontes"].get(h)
        if not det or "error" in det:
            continue
        tabla = det["importancia"].copy()
        tabla.insert(0, "Horizonte", f"+{h} h")
        tablas.append(tabla)

    if tablas:
        display(pd.concat(tablas, ignore_index=True).round(4))


## 6. Resumen para todas las variables

In [ ]:
filas = []
for target in TARGETS:
    m = entrenar_modelo(target, BASE_YEAR)
    if "error" in m:
        filas.append(
            {
                "Variable": TARGETS[target]["label"],
                "Horizonte": "—",
                "MAE": np.nan,
                "RMSE": np.nan,
                "R²": np.nan,
                "Nota": m["error"],
            }
        )
        continue

    for h, det in m["horizontes"].items():
        if "error" in det:
            filas.append(
                {
                    "Variable": TARGETS[target]["label"],
                    "Horizonte": f"+{h} h",
                    "MAE": np.nan,
                    "RMSE": np.nan,
                    "R²": np.nan,
                    "Nota": det["error"],
                }
            )
        else:
            filas.append(
                {
                    "Variable": TARGETS[target]["label"],
                    "Horizonte": f"+{h} h",
                    "MAE": det["mae"],
                    "RMSE": det["rmse"],
                    "R²": det["r2"],
                    "Nota": "",
                }
            )

resumen_modelos = pd.DataFrame(filas)
display(resumen_modelos.round(3))


## 7. Exportar resultados (opcional)

In [ ]:
# Descomenta para guardar los resultados en la carpeta del notebook.
# resumen_modelos.to_csv("resultados_modelo_predictivo.csv", index=False)

# for h, det in modelo.get("horizontes", {}).items():
#     if "error" not in det:
#         det["detalle"].to_csv(
#             f"predicciones_{TARGET}_{BASE_YEAR}_+{h}h.csv", index=False
#         )


## Notas y limitaciones

- El año base no participa en el entrenamiento; se usa únicamente para evaluar.
- 2015–2022 no tienen temperatura horaria útil y 2019 aparece prácticamente vacío.
- En 2023–2024 se usan PM₁₀ y PM₂.₅ NowCast; en años anteriores, las columnas crudas.
- CO8 se calcula como media móvil de 8 horas con al menos 6 lecturas; en 2023–2024 se usa el valor precalculado cuando falta.
- El modelo es autorregresivo y usa meteorología observada. No pronostica temperatura, humedad ni viento: si no se dispone de esas variables futuras, se asume persistencia o se usan valores disponibles.
- Por compatibilidad con el dashboard, las predicciones se recortan a cero. Si modelas temperatura y esperas valores negativos, ajusta la línea `np.maximum(0.0, ...)`.
- Para producción conviene hacer un backtesting temporal con varios años base y revisar deriva, valores extremos y ausencia de sensores.
